# ESCM 2.0 — exact spike-train comparison and inhibitory-history stress test

**Research question.** The original counter-based inhibition and the new additive inhibitory trace produced identical *spike counts* for four short input protocols. Are their **individual output spikes** and inhibitory trajectories also identical? When do the mechanisms diverge?

We compare the corrected ESCM 1.x baseline with **one** ESCM 2.0 candidate, changing **only** the inhibitory update:

- **Baseline:** neighboring spike increments the counter; the inhibitory state is *assigned* \(\mathrm{Inh}_i\leftarrow P_i W_k C_i\). The counter resets when neuron \(i\) fires.
- **Candidate:** neighboring spike *adds* to the inhibitory state, \(\mathrm{Inh}_i\leftarrow\mathrm{Inh}_i+P_i W_k\). No counter is used.

Both models retain the same membrane equation, self-excitation, persistence gate, clock, input stimulus, and complete square-neighborhood connectivity. We use exact output spike **(tick, neuron)** pairs for the primary comparison, plus membrane and inhibition state traces for mechanism diagnosis.

A discrepancy is **not** automatically a failure: the inhibitory laws are mathematically different. A match over tested inputs also does not establish general equivalence.

## 1. Reproducible protocols and discriminating inputs

A 5 × 5 network is sufficient to isolate the mechanism, and it preserves the original self-feedback implemented via outgoing recurrent pathways. All stimuli use integer multiples of the fixed 0.5 ms timestep.

Beyond the earlier four protocols, we vary neighbor arrival delay, repeatedly activate the neighbor, and move a single active pixel across three adjacent locations. These conditions probe how the baseline counter retains history **since the postsynaptic neuron's last spike**, whereas the candidate inhibitory current retains exponentially decaying history.

The nonspiking center's counter can grow without affecting its membrane potential while \(P=0\). That is a useful **diagnostic** of the bookkeeping but is not itself evidence of a behavioral difference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from brian2 import *
from tqdm.auto import tqdm

prefs.codegen.target = "numpy"
width = height = 5
N = width * height
center = 2 * width + 2
neighbor = center + 1
clock_s = 0.0005
end_ms = 60.0
k = 8

stimuli = {
    "single": ([center], [1.0]),
    "repeated_center": ([center, center, center], [1.0, 5.0, 9.0]),
    "sequential_4ms": ([center, neighbor], [1.0, 5.0]),
    "simultaneous": ([center, neighbor], [1.0, 1.0]),
}
for delay in (0.5, 1.0, 2.0, 8.0, 16.0):
    stimuli[f"neighbor_delay_{delay:g}ms"] = (
        [center, neighbor], [1.0, 1.0 + delay]
    )
stimuli["neighbor_burst"] = (
    [center, neighbor, neighbor, neighbor],
    [1.0, 5.0, 7.0, 9.0],
)
stimuli["moving_three"] = (
    [center - 1, center, neighbor], [1.0, 5.0, 9.0]
)

# Check the input protocol before constructing either network.
for name, (ids, times) in stimuli.items():
    ticks = np.rint(np.asarray(times) / (clock_s * 1000)).astype(int)
    assert np.allclose(ticks * clock_s * 1000, times), name
    assert len(set(zip(ticks, ids))) == len(ids), name
print("Protocols:", len(stimuli), " | Brian2 clock:", clock_s * 1000, "ms")
print("Parameters: threshold=0.1, input=0.5, self=6, lateral=-3, tau=0.2 ms")

## 2. Construct and run both variants

The unchanged continuous dynamics are

\[
\tau\dot v_i=-v_i+\mathrm{ExtIn}_i+P_i(\mathrm{Exc}_i+\mathrm{Inh}_i),\qquad
\tau\dot z_i=-z_i\quad (z\in\{\mathrm{ExtIn},\mathrm{Exc},\mathrm{Inh}\}).
\]

We preserve the corrected neuron-local counter reset, original custom-event conditions, and the recurrent pathway ordering. For the candidate, the counter is absent from the inhibition rule, but kept as a recorded, unused state to make model outputs directly comparable.

The use of a **group-specific parameter namespace**, with names distinct from notebook variables, avoids the ambiguous-name warnings seen in notebook 06.

In [ ]:
def simulate(mode, ids, times_ms):
    start_scope()
    defaultclock.dt = clock_s * second

    params = {
        "tau_model": 0.2 * ms, "threshold_model": 0.1,
        "reset_model": 0.0, "input_weight": 0.5,
        "self_weight": 6.0, "lateral_weight": -3.0,
    }
    camera = SpikeGeneratorGroup(
        N, np.asarray(ids, dtype=int), np.asarray(times_ms) * ms
    )
    eqs = """
    dv/dt = (-v + ExtIn + P*(Exc + Inh))/tau_model : 1
    dExtIn/dt = -ExtIn/tau_model : 1
    dExc/dt = -Exc/tau_model : 1
    dInh/dt = -Inh/tau_model : 1
    incoming_spikes : integer
    P : integer
    """
    neurons = NeuronGroup(
        N, eqs, threshold="v > threshold_model",
        reset="v = reset_model; incoming_spikes = 0",
        refractory=0*ms,
        events={
            "P_ON": "v > threshold_model",
            "P_OFF": "(timestep(t - lastspike, dt) > timestep(dt, dt) and v <= threshold_model)",
        },
        method="exact", namespace=params,
    )
    neurons.set_event_schedule("P_ON", when="after_thresholds")
    neurons.run_on_event("P_ON", "P = 1", when="after_thresholds")
    neurons.set_event_schedule("P_OFF", when="groups")
    neurons.run_on_event("P_OFF", "P = 0", when="groups")

    feed = Synapses(
        camera, neurons, on_pre="ExtIn_post = input_weight", namespace=params
    )
    feed.connect(j="i")

    if mode == "baseline":
        on_pre = {
            "pre": "incoming_spikes_post += 1; Exc_pre = self_weight",
            "pre_2": "Inh_post = P_post * lateral_weight * incoming_spikes_post",
        }
    else:
        on_pre = {
            "pre": "Exc_pre = self_weight",
            "pre_2": "Inh_post += P_post * lateral_weight",
        }
    recurrent = Synapses(neurons, neurons, on_pre=on_pre, namespace=params)
    recurrent.pre.order = 0
    recurrent.pre_2.order = 1

    source = np.repeat(np.arange(N), N)
    target = np.tile(np.arange(N), N)
    near = (
        (source != target)
        & (np.abs(source % width - target % width) <= k)
        & (np.abs(source // width - target // width) <= k)
    )
    recurrent.connect(i=source[near], j=target[near])

    spikes = SpikeMonitor(neurons)
    states = StateMonitor(
        neurons, ["v", "P", "Exc", "Inh", "incoming_spikes"],
        record=[center, neighbor], dt=defaultclock.dt,
    )
    net = Network(camera, neurons, feed, recurrent, spikes, states)
    net.run(end_ms * ms)
    return {
        "ticks": np.rint(np.asarray(spikes.t / second) / clock_s).astype(np.int64),
        "ids": np.asarray(spikes.i, dtype=np.int64),
        "time_ms": np.asarray(states.t / ms),
        "state": {key: np.asarray(getattr(states, key)).copy()
                  for key in ("v", "P", "Exc", "Inh", "incoming_spikes")},
    }

results = {}
for name, (ids, times) in tqdm(stimuli.items(), total=len(stimuli), desc="Stimuli"):
    results[name] = {
        "baseline": simulate("baseline", ids, times),
        "candidate": simulate("candidate", ids, times),
    }

## 3. Exact outputs and state differences

The strongest numerical test here is a comparison of **ordered output pairs** \((\mathrm{tick},\mathrm{neuron})\). We report the first differing spike, not merely whether total counts agree.

For each protocol we also measure the largest inhibitory-state difference for the two tracked neurons and the number of output spikes in the **input-free 20–60 ms interval**. Same spike train with different internal states would mean *functional equality on that input*, not mathematical equivalence.

No automatic PASS is reported for candidate behavior; this is a scientific comparison of two genuinely different rules.

In [ ]:
summary = []
for name, (ids, times) in stimuli.items():
    a, b = results[name]["baseline"], results[name]["candidate"]
    spikes_a = np.column_stack((a["ticks"], a["ids"]))
    spikes_b = np.column_stack((b["ticks"], b["ids"]))
    exact = np.array_equal(spikes_a, spikes_b)

    mismatch = None
    if not exact:
        common = min(len(spikes_a), len(spikes_b))
        mismatch = next(
            (i for i in range(common) if not np.array_equal(spikes_a[i], spikes_b[i])),
            common,
        )
    inh_delta = np.max(np.abs(a["state"]["Inh"] - b["state"]["Inh"]))
    persistent_a = np.count_nonzero(a["ticks"] * clock_s * 1000 >= 20)
    persistent_b = np.count_nonzero(b["ticks"] * clock_s * 1000 >= 20)
    summary.append((name, len(spikes_a), len(spikes_b), exact, mismatch,
                    inh_delta, persistent_a, persistent_b))

print(f"{'Protocol':23s} {'Spikes B/C':>12s} {'Exact':>7s} {'First diff':>10s} {'max |ΔInh|':>12s} {'Late B/C':>10s}")
print("-"*83)
for name, na, nb, exact, mismatch, delta, la, lb in summary:
    print(f"{name:23s} {na:5d}/{nb:<6d} {str(exact):>7s} {str(mismatch):>10s} {delta:12.3g} {la:4d}/{lb:<5d}")

print("\nFirst differing output spikes (if present):")
for name, na, nb, exact, mismatch, *_ in summary:
    if exact:
        continue
    a, b = results[name]["baseline"], results[name]["candidate"]
    for mode, r in (("baseline", a), ("candidate", b)):
        if mismatch < len(r["ids"]):
            print(f"  {name:23s} {mode:9s}: tick={r['ticks'][mismatch]}, neuron={r['ids'][mismatch]}")
        else:
            print(f"  {name:23s} {mode:9s}: no further spikes")

## 4. Visualize the hardest cases

We focus on repeated neighboring input and a moving three-pixel stimulus. Spikes are plotted for *all* neurons; state traces show only the center and right neighbor, where the counter's historical effects should be easiest to interpret.

The inhibitory-state comparison is particularly diagnostic: additive inhibition integrates a **decaying sum of past spikes**, while counter-based assignment uses a **discrete count since the last postsynaptic spike**. If their outputs diverge, these traces should help explain why.

In [ ]:
cases = ("neighbor_burst", "moving_three")
fig, axes = plt.subplots(len(cases), 2, figsize=(12, 7),
                         sharex=True, sharey=True, constrained_layout=True)
for row, name in enumerate(cases):
    for col, mode in enumerate(("baseline", "candidate")):
        ax = axes[row, col]
        r = results[name][mode]
        ax.scatter(r["ticks"] * clock_s * 1000, r["ids"], s=14, alpha=0.75)
        ids, times = stimuli[name]
        ax.scatter(times, ids, marker="x", color="tab:red", s=75, linewidths=1.8)
        ax.axvline(max(times), color="0.3", linestyle="--", linewidth=1)
        ax.set(title=f"{name}: {mode}", ylabel="Neuron index" if col == 0 else "",
               xlabel="Time (ms)" if row == len(cases)-1 else "",
               xlim=(0, end_ms), ylim=(0, N))
plt.show()

name = "neighbor_burst"
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, constrained_layout=True)
for ax, index, label in zip(axes, (0, 1), ("Old center", "New neighbor")):
    for mode in ("baseline", "candidate"):
        r = results[name][mode]
        ax.plot(r["time_ms"], r["state"]["Inh"][index], label=mode)
    for onset in stimuli[name][1]:
        ax.axvline(onset, color="0.75", linewidth=1, linestyle=":")
    ax.set(ylabel="Inhibitory trace (model units)", title=label)
    ax.grid(alpha=0.2)
axes[0].legend()
axes[-1].set_xlabel("Time (ms)")
plt.show()

## 5. Interpretation and decision

**Candidate acceptance should be functional, not solely numerical.** Ask whether it reproduces the baseline's two central properties:

1. One triggered location stays active without further camera events.
2. A newer neighboring location suppresses the old location while retaining its own activity.

For *implementation-equivalent* rewrites, exact output spikes should agree. The candidate's inhibitory rule is **not** implementation-equivalent, so differences must be examined in persistence and suppression terms.

**Scope of evidence:** Eleven hand-selected protocols, 25 neurons, 0.5 ms steps, one parameter set. No claims about the whole Event-Human3.6m recording or hardware resource savings follow from this comparison. If the candidate is promising, the next experiment should extend to the multi-pixel moving edge and then real recorded data. If it fails, vary one inhibitory parameter at a time, without simultaneously changing the gate or self-excitation.